# Laboratorio de Bases de Datos II
## Parte II: Citus en PostgreSQL - Dataset de Noticias sobre el Presidente Milei

Este Jupyter Notebook contiene la resolución completa de la Parte II del laboratorio, incluyendo:
- Descarga, limpieza y consolidación del dataset usando `kagglehub` y `pandas`.
- Inicialización de la base de datos `news_analysis_pg` y la distribución de la tabla en Citus.
- Creación de índices avanzados (GIN, B+Tree, Hash).
- Ejecución de consultas analíticas optimizadas (Q1 a Q5) con `EXPLAIN ANALYZE`.
- Script en Python para la medición secuencial de latencias de consultas.
- Respuestas analíticas al Cuestionario Teórico (Preguntas P8 y P10).

## 1. Descarga, Limpieza e Ingesta de Datos

El siguiente bloque de código realiza las siguientes acciones:
1. Descarga el dataset `torrezmn/milei-news` de Kaggle utilizando `kagglehub`.
2. Recorre recursivamente los directorios buscando archivos JSON.
3. Lee los archivos, maneja la estructura particular de múltiples objetos JSON por archivo, y limpia/consolida la información en un DataFrame.
4. Conecta al nodo coordinador de Citus.
5. Crea la tabla `milei_news`, la distribuye en Citus mediante la shard key `section`, y realiza una inserción masiva y de alto rendimiento utilizando `COPY`.

In [1]:
print("Cargando librerías...", flush=True)
import os
import json
import io
from pathlib import Path
import pandas as pd
import psycopg2
import kagglehub
from tqdm import tqdm
print("Librerías cargadas correctamente.", flush=True)

# Configuración de conexión al nodo coordinador de Citus (ejecutándose en local por puerto 5432)
DB_CONFIG = {
    "dbname": "news_analysis_pg",
    "user": "postgres",
    "password": "password",
    "host": "localhost",
    "port": 5432
}
TABLE_NAME = "milei_news"

# 1. Verificar si el dataset ya está localmente para evitar esperas de red y errores en Windows
dataset_path = Path("data/Argentina")
if not dataset_path.exists():
    dataset_path = Path("../data/Argentina")

if dataset_path.exists():
    print(f"Dataset local encontrado en: {dataset_path.absolute()}")
else:
    print("Dataset local no encontrado. Intentando descargar desde Kaggle...")
    try:
        dataset_path_str = kagglehub.dataset_download("torrezmn/milei-news")
        dataset_path = Path(dataset_path_str)
        print("Dataset descargado con éxito en:", dataset_path.absolute())
    except Exception as e:
        raise FileNotFoundError(
            f"No se pudo descargar el dataset ni se encontró la carpeta local en {dataset_path.absolute()}. Error: {e}"
        )

# 2. Funciones auxiliares de parseo, normalización y limpieza
def normalize_column_name(col):
    return col.strip().lower().replace(" ", "_").replace("-", "_")

def parse_concatenated_json(text):
    decoder = json.JSONDecoder()
    pos = 0
    objs = []
    text = text.strip()
    while pos < len(text):
        while pos < len(text) and text[pos].isspace():
            pos += 1
        if pos >= len(text):
            break
        try:
            obj, idx = decoder.raw_decode(text, pos)
            if isinstance(obj, list):
                objs.extend(obj)
            else:
                objs.append(obj)
            pos = idx
        except json.JSONDecodeError:
            next_bracket = text.find('{', pos + 1)
            next_array = text.find('[', pos + 1)
            indices = [idx for idx in [next_bracket, next_array] if idx != -1]
            if not indices:
                break
            pos = min(indices)
    return objs

def clean_dataframe(df):
    df.columns = [normalize_column_name(c) for c in df.columns]
    expected_columns = ["news_paper", "section", "title", "summary", "published", "link", "tags"]
    
    for col in expected_columns:
        if col not in df.columns:
            df[col] = None
            
    df["news_paper"] = df["news_paper"].fillna("unknown").astype(str).str.strip()
    df["section"] = df["section"].fillna("unknown").astype(str).str.lower().str.strip()
    df["title"] = df["title"].fillna("").astype(str).str.strip()
    df["summary"] = df["summary"].fillna("").astype(str).str.strip()
    df["link"] = df["link"].fillna("").astype(str).str.strip()
    df["tags"] = df["tags"].fillna("").astype(str).str.strip()
    
    df["published"] = pd.to_datetime(df["published"], errors="coerce", utc=True)
    df = df[~((df["title"] == "") & (df["summary"] == ""))]
    return df[expected_columns]

# 3. Recorrer y leer archivos recursivamente de forma rápida en memoria
all_records = []
json_files = list(dataset_path.rglob("*.json"))
print(f"Archivos JSON encontrados: {len(json_files)}")

for idx, json_path in enumerate(tqdm(json_files, desc="Leyendo JSONs")):
    try:
        parts = json_path.parts
        if len(parts) < 3:
            continue
        news_paper = parts[-3]
        section = parts[-2]
        with open(json_path, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
        records = parse_concatenated_json(content)
        if records:
            for r in records:
                if isinstance(r, dict):
                    r["news_paper"] = news_paper
                    r["section"] = section
                    all_records.append(r)
    except Exception as e:
        pass

print(f"Total de registros decodificados: {len(all_records)}")
print("Creando DataFrame y aplicando limpieza...")
df_consolidated = pd.DataFrame(all_records)
df_consolidated = clean_dataframe(df_consolidated)
print(f"DataFrame consolidado final listo con dimensiones: {df_consolidated.shape}")

# 4. Carga a Citus en PostgreSQL
print("Conectando a Citus coordinator...")
conn = psycopg2.connect(**DB_CONFIG)
cursor = conn.cursor()
try:
    # 1. Habilitar la extensión de Citus si no está habilitada
    print("Habilitando extensión Citus...")
    cursor.execute("CREATE EXTENSION IF NOT EXISTS citus;")
    conn.commit()

    # 2. Registrar los nodos trabajadores (workers) si no están ya registrados
    print("Verificando nodos trabajadores...")
    cursor.execute("SELECT * FROM pg_dist_node;")
    nodes = cursor.fetchall()
    if not nodes:
        print("Registrando nodos trabajadores ('worker1' y 'worker2')...")
        cursor.execute("SELECT citus_add_node('worker1', 5432);")
        cursor.execute("SELECT citus_add_node('worker2', 5432);")
        conn.commit()
        print("Nodos trabajadores registrados.")
    else:
        print(f"Nodos ya registrados: {[n[1] for n in nodes]}")

    # 3. Crear tabla de destino
    cursor.execute(f"""
        CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
            news_paper VARCHAR(100),
            section VARCHAR(100),
            title TEXT,
            summary TEXT,
            published TIMESTAMP,
            link TEXT,
            tags TEXT
        );
    """)
    conn.commit()
    print("Tabla creada o verificada.")

    # 4. Verificar si está distribuida, de lo contrario distribuirla
    cursor.execute("SELECT 1 FROM pg_dist_partition WHERE logicalrelid = %s::regclass;", (TABLE_NAME,))
    if not cursor.fetchone():
        print(f"Distribuyendo tabla '{TABLE_NAME}' por shard key 'section'...")
        cursor.execute(f"SELECT create_distributed_table('{TABLE_NAME}', 'section');")
        conn.commit()
        print("Tabla distribuida exitosamente.")
    else:
        print("La tabla ya está distribuida en el clúster Citus.")
    
    # Carga masiva usando COPY (rápido y optimizado)
    print("Limpiando datos previos e iniciando carga masiva...")
    cursor.execute(f"TRUNCATE TABLE {TABLE_NAME};")
    conn.commit()

    df_copy = df_consolidated.copy()
    if pd.api.types.is_datetime64_any_dtype(df_copy["published"]):
        df_copy["published"] = df_copy["published"].dt.strftime("%Y-%m-%d %H:%M:%S")
    df_copy = df_copy.where(pd.notnull(df_copy), None)

    buffer = io.StringIO()
    df_copy.to_csv(buffer, sep=",", header=False, index=False, na_rep="\\N")
    buffer.seek(0)

    cursor.copy_expert(f"COPY {TABLE_NAME} FROM STDIN WITH (FORMAT csv, NULL '\\N')", buffer)
    conn.commit()
    print("Carga masiva por COPY completada exitosamente.")

    # Verificar conteo
    cursor.execute(f"SELECT COUNT(*) FROM {TABLE_NAME};")
    total_rows = cursor.fetchone()[0]
    print(f"Total de registros cargados en base de datos distribuida: {total_rows}")
finally:
    cursor.close()
    conn.close()


Cargando librerías...
Librerías cargadas correctamente.


c:\Users\jorge\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset local encontrado en: c:\Users\jorge\Desktop\Proyectos\Lab15-BD2\citus\..\data\Argentina
Archivos JSON encontrados: 23495


Leyendo JSONs: 100%|██████████| 23495/23495 [00:03<00:00, 5881.93it/s]


Total de registros decodificados: 51973
Creando DataFrame y aplicando limpieza...
DataFrame consolidado final listo con dimensiones: (51874, 7)
Conectando a Citus coordinator...
Habilitando extensión Citus...
Verificando nodos trabajadores...
Nodos ya registrados: [1, 2]
Tabla creada o verificada.
La tabla ya está distribuida en el clúster Citus.
Limpiando datos previos e iniciando carga masiva...
Carga masiva por COPY completada exitosamente.
Total de registros cargados en base de datos distribuida: 51874


## 2. Indexación

Para optimizar el rendimiento de las consultas analíticas de búsqueda textual, filtros temporales y agrupaciones, creamos los siguientes índices:
1. **Índice GIN (Generalized Inverted Index)** sobre el vector de búsqueda textual en español para la combinación de `title` y `summary`.
2. **Índice B+Tree** sobre la fecha de publicación (`published`) ordenada de forma descendente.
3. **Índice Hash** sobre el periódico (`news_paper`) para búsquedas de igualdad directa.

Citus propaga automáticamente estos comandos a todos los shards en los nodos workers.

In [2]:
conn = psycopg2.connect(**DB_CONFIG)
cursor = conn.cursor()
try:
    print("Creando índice GIN (FTS) sobre title || ' ' || summary...")
    cursor.execute("""
        CREATE INDEX IF NOT EXISTS idx_milei_news_fts 
        ON milei_news USING GIN (to_tsvector('spanish', title || ' ' || summary));
    """)
    conn.commit()

    print("Creando índice B+Tree sobre la fecha de publicación...")
    cursor.execute("""
        CREATE INDEX IF NOT EXISTS idx_milei_news_published 
        ON milei_news USING btree (published DESC NULLS LAST);
    """)
    conn.commit()

    print("Creando índice Hash sobre news_paper...")
    cursor.execute("""
        CREATE INDEX IF NOT EXISTS idx_milei_news_newspaper 
        ON milei_news USING hash (news_paper);
    """)
    conn.commit()
    print("Todos los índices se crearon correctamente.")
finally:
    cursor.close()
    conn.close()


Creando índice GIN (FTS) sobre title || ' ' || summary...
Creando índice B+Tree sobre la fecha de publicación...
Creando índice Hash sobre news_paper...
Todos los índices se crearon correctamente.


## 3. Consultas de Referencia (Q1 a Q5)

Ejecutamos las 5 consultas solicitadas utilizando `EXPLAIN ANALYZE` para evaluar el plan de ejecución física distribuida en Citus, viendo cómo interactúa el coordinador con los workers a través del nodo `Custom Scan (Citus)`.

In [3]:
def run_query_explain(query_name, sql):
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    try:
        print(f"=== EXPLAIN ANALYZE PARA: {query_name} ===\n")
        cursor.execute(sql)
        plan = cursor.fetchall()
        for line in plan:
            print(line[0])
        print("\n" + "="*80 + "\n")
    except Exception as e:
        print(f"Error ejecutando {query_name}: {e}")
    finally:
        cursor.close()
        conn.close()

# Q1: Búsqueda textual por múltiples términos ordenada por relevancia (ts_rank)
q1_sql = """
EXPLAIN ANALYZE
SELECT news_paper, section, title, summary, published, 
       ts_rank(to_tsvector('spanish', title || ' ' || summary), to_tsquery('spanish', 'dólar & inflación')) AS rank
FROM milei_news
WHERE to_tsvector('spanish', title || ' ' || summary) @@ to_tsquery('spanish', 'dólar & inflación')
ORDER BY rank DESC;
"""
run_query_explain("Q1 (Búsqueda textual: dólar & inflación)", q1_sql)

# Q2: Búsqueda con exclusión ("Milei" sin "Twitter") ordenada por fecha
q2_sql = """
EXPLAIN ANALYZE
SELECT news_paper, section, title, summary, published
FROM milei_news
WHERE to_tsvector('spanish', title || ' ' || summary) @@ to_tsquery('spanish', 'Milei & !Twitter')
ORDER BY published DESC;
"""
run_query_explain("Q2 (Búsqueda con exclusión: Milei & !Twitter)", q2_sql)

# Q3: Ordenamiento simple por fecha (10 noticias más recientes)
q3_sql = """
EXPLAIN ANALYZE
SELECT news_paper, section, title, summary, published
FROM milei_news
ORDER BY published DESC
LIMIT 10;
"""
run_query_explain("Q3 (Ordenamiento simple: top 10 recientes)", q3_sql)

# Q4: Agregación agrupada por clave de distribución (section)
q4_sql = """
EXPLAIN ANALYZE
SELECT section, COUNT(*) AS cantidad
FROM milei_news
GROUP BY section
ORDER BY cantidad DESC;
"""
run_query_explain("Q4 (Agregación por Shard Key)", q4_sql)

# Q5: Agregación agrupada por atributo no particionado (news_paper)
q5_sql = """
EXPLAIN ANALYZE
SELECT news_paper, COUNT(*) AS cantidad
FROM milei_news
GROUP BY news_paper
ORDER BY cantidad DESC;
"""
run_query_explain("Q5 (Agregación por Atributo No Shardeado)", q5_sql)


=== EXPLAIN ANALYZE PARA: Q1 (Búsqueda textual: dólar & inflación) ===

Sort  (cost=31205.32..31455.32 rows=100000 width=512) (actual time=71.281..71.284 rows=81 loops=1)
  Sort Key: remote_scan.rank DESC
  Sort Method: quicksort  Memory: 55kB
  ->  Custom Scan (Citus Adaptive)  (cost=0.00..0.00 rows=100000 width=512) (actual time=71.228..71.235 rows=81 loops=1)
        Task Count: 32
        Tuple data received from nodes: 28 kB
        Tasks Shown: One of 32
        ->  Task
              Tuple data received from node: 0 bytes
              Node: host=worker1 port=5432 dbname=news_analysis_pg
              ->  Seq Scan on milei_news_102016 milei_news  (cost=0.00..9.21 rows=1 width=283) (actual time=2.371..2.372 rows=0 loops=1)
                    Filter: (to_tsvector('spanish'::regconfig, ((title || ' '::text) || summary)) @@ '''dol'' & ''inflacion'''::tsquery)
                    Rows Removed by Filter: 26
                  Planning Time: 0.787 ms
                  Execution Time: 2

## 4. Script de Medición de Tiempos

El siguiente script en Python ejecuta las consultas Q1 a Q5 secuencialmente y sin `EXPLAIN` para registrar la latencia de red y ejecución real en milisegundos. Al finalizar, muestra los resultados tabulados en un DataFrame de Pandas.

In [ ]:
import time

queries = {
    "Q1 (Búsqueda textual: dólar inflación)": """
        SELECT news_paper, section, title, summary, published, 
               ts_rank(to_tsvector('spanish', title || ' ' || summary), to_tsquery('spanish', 'dólar & inflación')) AS rank
        FROM milei_news
        WHERE to_tsvector('spanish', title || ' ' || summary) @@ to_tsquery('spanish', 'dólar & inflación')
        ORDER BY rank DESC;
    """,
    "Q2 (Búsqueda con exclusión: Milei sin Twitter)": """
        SELECT news_paper, section, title, summary, published
        FROM milei_news
        WHERE to_tsvector('spanish', title || ' ' || summary) @@ to_tsquery('spanish', 'Milei & !Twitter')
        ORDER BY published DESC;
    """,
    "Q3 (Ordenamiento simple: 10 recientes)": """
        SELECT news_paper, section, title, summary, published
        FROM milei_news
        ORDER BY published DESC
        LIMIT 10;
    """,
    "Q4 (Agregación por shard key: count by section)": """
        SELECT section, COUNT(*) AS cantidad
        FROM milei_news
        GROUP BY section
        ORDER BY cantidad DESC;
    """,
    "Q5 (Agregación sin shard key: count by news_paper)": """
        SELECT news_paper, COUNT(*) AS cantidad
        FROM milei_news
        GROUP BY news_paper
        ORDER BY cantidad DESC;
    """
}

results = []

for q_name, q_sql in queries.items():
    print(f"Ejecutando {q_name}...")
    
    # Precalentamiento de la base de datos (evitar cold starts)
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    try:
        cursor.execute(q_sql)
        cursor.fetchall()
    except Exception as e:
        print(f"Error precalentando: {e}")
        cursor.close()
        conn.close()
        continue
        
    # Medición real
    start_time = time.time()
    cursor.execute(q_sql)
    rows = cursor.fetchall()
    end_time = time.time()
    
    latency_ms = (end_time - start_time) * 1000
    row_count = len(rows)
    
    results.append({
        "Consulta": q_name,
        "Latencia Real (ms)": f"{latency_ms:.2f}",
        "Filas Retornadas": row_count
    })
    
    cursor.close()
    conn.close()

# Tabular los resultados con Pandas
df_results = pd.DataFrame(results)
df_results


## 5. Cuestionario Teórico

### Pregunta 8 (P8): ¿Cómo funciona exactamente el índice GIN para la búsqueda textual en PostgreSQL? ¿La función ts_rank utiliza TF-IDF? ¿Cómo se compara este enfoque con el índice invertido que usa MongoDB?

#### 1. Funcionamiento del Índice GIN (Generalized Inverted Index)
El índice GIN es un **índice invertido generalizado** diseñado para manejar columnas con valores compuestos (arrays, documentos JSON, vectores de texto completo). En PostgreSQL, el flujo de funcionamiento para búsqueda de texto completo (FTS) es el siguiente:
- **Procesamiento de texto (`to_tsvector`)**: El texto de las columnas indicadas se divide en fichas (tokens), las cuales pasan por filtros de stop-words (palabras sin valor analítico como 'el', 'un', 'y') y son transformadas por diccionarios lingüísticos a sus formas base o raíces denominadas **lexemas**.
- **Estructura del Índice**: GIN mapea cada **lexema** único a una lista física de IDs de filas (*postings list* o un B-Tree interno de *TIDs - Tuple Identifiers*) en el disco en las cuales aparece dicho lexema.
- **Evaluación del Operador (`@@`)**: Cuando se ejecuta la búsqueda usando `to_tsquery('dólar & inflación')`, PostgreSQL busca los lexemas 'dolar' e 'inflacion' en la estructura del índice GIN, obtiene sus respectivas listas de TIDs y realiza operaciones binarias de conjuntos (en este caso una intersección `AND`) en memoria. El motor lee físicamente solo las páginas de datos que contienen los TIDs coincidentes, logrando un rendimiento sumamente alto y evitando búsquedas secuenciales sobre millones de registros.

#### 2. ts_rank y TF-IDF
**No, `ts_rank` no utiliza TF-IDF de forma global.** El esquema TF-IDF (Term Frequency - Inverse Document Frequency) penaliza las palabras que son muy comunes en todo el corpus global de la base de datos (IDF) y premia aquellas que son raras globalmente pero frecuentes localmente en el documento (TF).
PostgreSQL calcula `ts_rank` de forma **local por fila** para mantener el rendimiento dinámico de las consultas (no realiza análisis global del corpus completo en tiempo real). Los factores que mide `ts_rank` son:
- **Frecuencia del término (TF) local**: Cuántas veces aparece el lexema en el registro evaluado.
- **Proximidad física**: Cuán cerca se encuentran los lexemas de búsqueda entre sí (especialmente explotado por `ts_rank_cd`).
- **Pesos de estructura**: Permite asignar coeficientes de peso (`setweight`) predefinidos (A, B, C, D) a distintas columnas. Por defecto, A=1.0, B=0.4, C=0.2, D=0.1.

#### 3. Comparación con el índice invertido de MongoDB
- **Algoritmo de relevancia**: MongoDB implementa de forma nativa una puntuación basada en **TF-IDF** en sus consultas `$text`, escalando la relevancia local de los términos según su rareza global dentro de la colección. PostgreSQL requiere de infraestructura personalizada si se busca un comportamiento TF-IDF global estricto.
- **Flexibilidad de configuración**: PostgreSQL soporta múltiples configuraciones de búsqueda de texto completo y múltiples índices GIN sobre diferentes expresiones o columnas de una misma tabla. MongoDB restringe a **un solo índice de texto por colección**, por lo cual todos los campos textuales buscables deben integrarse dentro del mismo índice.
- **Capacidad lingüística**: PostgreSQL provee herramientas de diccionarios, sinónimos (`thesaurus`) y procesadores de lexemas avanzados, lo cual permite afinar la gramática del idioma (español en este caso) con alta precisión. MongoDB tiene opciones de stemmers más limitadas a nivel de motor SQL.

---

### Pregunta 10 (P10): ¿Cómo distribuye y paraleliza Citus cada tipo de consulta (búsqueda de texto vs agregaciones por shard key vs agregaciones sin shard key)? ¿De qué manera se puede detectar el overhead de coordinación revisando el nodo "Custom Scan (Citus)" en un EXPLAIN ANALYZE?

#### 1. Distribución y Paralelización por Tipo de Consulta
- **Búsqueda de texto (Q1, Q2)**:
  Al no filtrar explícitamente por la shard key `section` en el `WHERE`, el coordinador de Citus desconoce qué shard almacena la coincidencia. El planificador de Citus genera **tareas paralelas** para cada uno de los shards distribuidos y las despacha a todos los nodos workers. Los workers ejecutan las búsquedas localmente utilizando sus CPU y su respectivo índice GIN de forma concurrente, devolviendo los resultados parciales filtrados y ordenados. El coordinador finaliza recolectando y combinando los conjuntos antes de entregarlos al cliente.
- **Agregaciones por Shard Key (Q4)**:
  Dado que agrupamos por la clave de distribución (`GROUP BY section`), Citus asegura por diseño que todos los registros de una sección particular residen en el mismo shard y en el mismo nodo físico. Esto permite a Citus paralelizar completamente el `GROUP BY` y el `COUNT(*)` delegándolo al 100% de forma local a los workers (*co-located aggregation*). No hay necesidad de transferir filas o realizar barajados (shuffling) entre workers. El coordinador solo recolecta y concatena la lista final de resultados ya agregados.
- **Agregaciones sin Shard Key (Q5)**:
  Cuando agrupamos por `news_paper` (que no es la clave de distribución), las filas de un mismo periódico se encuentran distribuidas en todos los shards en múltiples workers. Citus se ve obligado a realizar una **agregación en dos fases**:
  1. *Fase Local*: Cada worker agrupa y calcula subtotales locales de conteo para cada `news_paper` presente en sus shards locales.
  2. *Fase Global*: Los workers envían los subtotales agregados al coordinador, el cual ejecuta un `SUM` final del conteo recibido para unificar las estadísticas de cada periódico.
  Esto añade carga de red y procesamiento centralizado en el coordinador.

#### 2. Detección del Overhead de Coordinación con EXPLAIN ANALYZE
En Citus, las consultas distribuidas se planifican en el coordinador mediante un nodo especial llamado `Custom Scan (Citus)`. Al ejecutar `EXPLAIN ANALYZE`, podemos identificar el overhead de coordinación de las siguientes formas:
- **Task Count**: Indica el número de sub-consultas TCP enviadas a los workers. Si este conteo es muy alto para consultas cortas, la latencia de red por establecer sockets dominará el tiempo total.
- **Remote SQL**: Muestra la consulta exacta enviada a cada worker, útil para evaluar si la agregación fue delegada eficientemente o si se arrastran filas crudas.
- **Comparación de tiempos (Coordinador vs Worker)**: Si comparamos el `Execution Time` global devuelto por el coordinador contra los tiempos de las tareas locales (`Task Execution Time` en el bucle del worker), notaremos una brecha de tiempo. Esta brecha corresponde al **overhead de coordinación** (latencia de red ida y vuelta, tiempo de serialización/deserialización de resultados, y tiempo utilizado por el coordinador para combinar y ordenar las filas recibidas antes de entregarlas al cliente).